# 05 — Controller v2: learned risk under a fixed INT8 budget (pre-registered follow-up)

**Purpose.** This is the follow-up to notebook 04 that responds to the review. The design is frozen
in `configs/prereg_addendum.yaml`, and its SHA-256 is checked before anything runs; the notebook
stops if the file changed. **The r04 result stays the pre-registered primary outcome.** v2 is
reported whether it succeeds or fails.
1. **Engines + calib cache** (rebuilt; gate vs committed Step 2).
2. **Calib cues** (Step 3 logic).
3. **Fit + lock on calib only:**
   - gradient-boosted risk model (clip-grouped CV) predicting the per-frame FP16-over-INT8 gain;
   - cross-fitted closed-loop thresholds for INT8 budgets 25 / 50 / 75 %, and K per budget;
   - v1 (r04 controller) re-thresholded to each budget;
   - lagged-oracle thresholds;
   - every lock hashed.
4. **Test cache**, only after the verified v2@50% lock.
5. **Energy under mixed switching** on 6 seeded test clips (FP16, INT8, v2 pipelined and sequential,
   forced alternation), plus a live-vs-replay check.
6. **Test evaluation, once:**
   - per budget: v2 vs matched random (10 seeds), v1, oracle (upper bound, uses GT) and lagged
     oracle (GT one frame late, not deployable);
   - cluster-bootstrap CIs; headroom captured; P1–P5.

**Before running**
1. *Add Data* → **`tcjpaperdatasets`**.
2. *Settings* → **Accelerator: GPU T4 x2**, **Internet: On**; *Add-ons → Secrets* → tick **`GH_TOKEN`**.
3. `SMOKE_TEST = True` first: about 30 min, mostly engine builds. Then `SMOKE_TEST = False` →
   *Save Version → Save & Run All*: about 2–2.5 h. These are estimates, not yet measured.

**Download afterwards:** `results_05_controller_v2.zip` → unzip into the repo root. It contains
`results/bdd100k_mot/r05_*`, `results/locked/locked_v2_*` / `locked_v1_*` / `locked_lagged_*`,
`v2_model.pkl` with its hash recorded in the locks, and `logs/`.


In [ ]:
# ===================== CELL 1: CONFIG + ENVIRONMENT CHECK =====================
# Edit nothing below for a normal run except SMOKE_TEST.
SMOKE_TEST = True   # True: tiny subset (run this first). False: full run.
NOTEBOOK_REV = "c8e9e5317d06"   # hash of this notebook's generated source; checked against the cloned repo

CONFIG = {
    "notebook": "05_controller_v2",
    "repo_https": "https://github.com/vinaykamble-nsio/adapres.git",       # private repo, cloned with Secret GH_TOKEN
    "repo_ref": "main",
    "repo_dir": "/tmp/adapres",                     # /tmp: not saved into notebook output
    "repo_zip_marker": "adapres/GIT_COMMIT",        # fallback: repo zip attached as a dataset
    "config_path": "configs/default.yaml",          # relative to repo_dir
    "overrides": {                                  # applied on top of default.yaml
        "paths": {"input_root": "/kaggle/input", "work_root": "/kaggle/working"},
    },
    "cuda_visible_devices": "0",                    # T4 x2 -> pin to one GPU
    "required_gpu_substring": "T4",
}

import os, sys, subprocess, json, platform, base64, glob, shutil, time
os.environ["CUDA_VISIBLE_DEVICES"] = CONFIG["cuda_visible_devices"]  # before any CUDA import
T0 = time.time()

def sh(cmd, check=True):
    """Run a shell command, print its output, raise with the real error on failure."""
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)
    print(r.stdout[-6000:])
    if r.stderr.strip():
        print(r.stderr[-6000:], file=sys.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r

sh("nvidia-smi")
print("python", sys.version)
for mod in ("torch", "onnxruntime", "tensorrt", "ultralytics"):
    try:
        m = __import__(mod)
        print(f"{mod} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{mod} NOT AVAILABLE YET ({e!r}); installed in cell 2 if pinned")
import torch
print("torch.cuda.is_available:", torch.cuda.is_available(), "| visible devices:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2.")
gpu = torch.cuda.get_device_name(0)
print("GPU 0:", gpu)
if CONFIG["required_gpu_substring"] not in gpu:
    raise RuntimeError(f"GPU is {gpu!r}; this project requires a T4 (P100 has no fast INT8). "
                       "Settings -> Accelerator -> GPU T4 x2, then restart.")
if torch.cuda.device_count() != 1:
    raise RuntimeError(f"expected exactly 1 visible GPU after pinning, got {torch.cuda.device_count()}")

# ---- get the repo: private GitHub clone (Secret GH_TOKEN), else attached zip dataset
REPO = CONFIG["repo_dir"]
repo_source = None
if os.path.isdir(REPO):
    shutil.rmtree(REPO)
try:
    if "github.com/OWNER/" in CONFIG["repo_https"]:
        raise RuntimeError("repo_https is still the placeholder: set it in configs/kaggle.yaml and regenerate")
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    if not token:
        raise RuntimeError("Secret GH_TOKEN is empty")
    hdr = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    # token passed as a one-off header: never written to .git/config or printed
    r = subprocess.run(["git", "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {hdr}",
                        "clone", "--quiet", CONFIG["repo_https"], REPO], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone failed: " + r.stderr.replace(token, "***").replace(hdr, "***"))
    sh(["git", "-C", REPO, "checkout", "--quiet", CONFIG["repo_ref"]])
    repo_source = "github"
    del token, hdr
except Exception as e:
    print(f"GitHub clone unavailable: {e!r}\nTrying zip-dataset fallback ...")
    hits = sorted(glob.glob("/kaggle/input/**/" + CONFIG["repo_zip_marker"], recursive=True))
    if not hits:
        raise RuntimeError("Neither GitHub clone (Secret GH_TOKEN) nor an attached repo zip "
                           "(dataset containing adapres/GIT_COMMIT) is available. See README cell.") from e
    shutil.copytree(os.path.dirname(hits[0]), REPO)
    repo_source = "zip_dataset:" + os.path.dirname(hits[0])

sys.path.insert(0, os.path.join(REPO, "src"))
sys.path.insert(0, os.path.join(REPO, "scripts"))
# Re-running cells in a live kernel re-clones the repo but Python would keep the OLD imported
# modules while GIT records the NEW commit. Drop them so the code always matches the hash.
_stale = [m for m in list(sys.modules) if m == "adapres" or m.startswith("adapres.") or m == "make_notebooks"]
for m in _stale:
    del sys.modules[m]
if _stale:
    print(f"dropped {len(_stale)} previously imported adapres modules (fresh code from this clone)")
import make_notebooks
expected_rev = make_notebooks.notebook_rev(CONFIG["notebook"] + ".ipynb")
if expected_rev != NOTEBOOK_REV:
    raise RuntimeError(f"STALE NOTEBOOK: this notebook is rev {NOTEBOOK_REV} but repo {CONFIG['repo_ref']} "
                       f"generates rev {expected_rev}. Re-import notebooks/{CONFIG['notebook']}.ipynb "
                       "from the repo (File -> Import Notebook), or push your local commits.")
print("notebook rev", NOTEBOOK_REV, "matches repo")
from adapres import config as C
from adapres.repro import git_commit, set_seed
cfg = C.load(os.path.join(REPO, CONFIG["config_path"]), CONFIG["overrides"])
SEED = cfg["seed"]
set_seed(SEED)
GIT = git_commit(REPO)
print("repo source:", repo_source)
print("git commit:", GIT)
print("seed:", SEED, "| SMOKE_TEST:", SMOKE_TEST)
if GIT["commit"] is None:
    raise RuntimeError("could not determine the git commit; refusing to produce unattributable results")
if GIT["dirty"]:
    raise RuntimeError("repo checkout is dirty; results would not match the recorded commit")
RESULTS = C.resolve_out(cfg, "results_dir", SMOKE_TEST)
LOGS = C.resolve_out(cfg, "logs_dir", SMOKE_TEST)
# Start every run with empty results/ and logs/: the final zip packs these folders, and leftovers
# from an earlier run in the same session would otherwise be shipped as if produced now.
# (Resumable caches live in work dirs, not here.)
for d in (RESULTS, LOGS):
    if os.path.isdir(d) and os.listdir(d):
        print(f"clearing stale outputs from an earlier run in this session: {d} ({len(os.listdir(d))} entries)")
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)
print("results ->", RESULTS, "| logs ->", LOGS)


In [ ]:
# ===================== CELL 2: LOCKED INSTALLS + VERSION CHECK =====================
# Internet must be ON. Installs only what the Kaggle image lacks; torch is not touched.
# Every pin is exact (locked 2026-11-20); the cell fails if anything installs at another version.
import re, importlib.metadata as md
print("python", sys.version, "| torch", torch.__version__, "| torch CUDA", torch.version.cuda)
REQ = os.path.join(REPO, "requirements-kaggle.txt")
lines = [ln.split("#")[0].strip() for ln in open(REQ)]
opts = [tok for ln in lines if ln.startswith("-") for tok in ln.split()]  # e.g. --extra-index-url URL
print("pip options:", opts)
reqs = [ln for ln in lines if ln and not ln.startswith("-")]
PIP = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check"]
# Preflight: resolve each requirement on its own (no install) so ALL failures show in one run.
bad = {}
for req in reqs:
    p = subprocess.run(PIP + ["--dry-run", "--no-deps", "--quiet", *opts, req], capture_output=True, text=True)
    if p.returncode != 0:
        bad[req] = "\n".join(l for l in p.stderr.splitlines() if "ERROR" in l) or p.stderr[-2000:]
    print(f"preflight {req:45s} {'OK' if p.returncode == 0 else 'FAILED'}")
if bad:
    for req, err in bad.items():
        print(f"\n--- {req} ---\n{err}")
    raise RuntimeError(f"{len(bad)} requirement(s) have no wheel for this Python/CUDA: {list(bad)}. "
                       "Paste this whole cell output; the available versions are listed above.")
r = sh(PIP + ["--quiet", "-r", REQ], check=False)
with open(os.path.join(LOGS, f"{CONFIG['notebook']}_pip.log"), "w") as f:
    f.write(r.stdout + "\n--- stderr ---\n" + r.stderr)
if r.returncode != 0:
    raise RuntimeError("pip install failed (real error above).")
importlib.invalidate_caches()
# Version lock: record exactly what got installed so requirements-kaggle.txt can be pinned to it.
names = [re.split(r"[<>=!~\[ ]", q)[0] for q in reqs]
resolved = {n: md.version(n) for n in names}
os.makedirs(os.path.join(RESULTS, "env"), exist_ok=True)
lock = "\n".join(f"{n}=={v}" for n, v in resolved.items())
with open(os.path.join(RESULTS, "env", "pip_resolved.txt"), "w") as f:
    f.write(f"# python {sys.version.split()[0]} torch {torch.__version__} cuda {torch.version.cuda}\n{lock}\n")
print("RESOLVED VERSIONS (copy into requirements-kaggle.txt as exact pins):\n" + lock)
trt_dists = ("tensorrt-cu13", "tensorrt-cu13-bindings", "tensorrt-cu13-libs")
if len({resolved.get(d) for d in trt_dists}) != 1:
    raise RuntimeError(f"TensorRT wrapper/bindings/libs version mismatch: {resolved}")
pinned = {re.split(r"[<>=!~\[ ]", q)[0]: q.split("==")[-1] for q in reqs if "==" in q}
drift = {n: (pinned[n], resolved[n]) for n in pinned if pinned[n] != resolved[n]}
if drift:
    raise RuntimeError(f"installed versions differ from the lock (pinned, installed): {drift}")
try:
    import tensorrt
except ImportError:
    # Diagnose instead of guessing: which top-level modules did each TensorRT dist install?
    for d in trt_dists:
        try:
            tops = sorted({str(f).split("/")[0] for f in (md.files(d) or [])})
            print(f"{d}: top-level entries {tops[:25]}")
        except md.PackageNotFoundError:
            print(f"{d}: NOT INSTALLED")
    raise
import onnx, onnxruntime
print("import OK: tensorrt", tensorrt.__version__, "| onnx", onnx.__version__, "| onnxruntime", onnxruntime.__version__)
print("torch still", torch.__version__, "cuda", torch.version.cuda)


In [ ]:
# ===================== CELL 3: SETUP (weights, run object, fail-safe zip) =====================
import logging, zipfile
os.environ["YOLO_AUTOINSTALL"] = "False"
logging.basicConfig(level=logging.INFO, force=True, format="%(asctime)s %(levelname)s %(name)s: %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(os.path.join(LOGS, "05_v2.log"))])
from adapres.pipeline.full import GateError
from adapres.pipeline.v2 import V2Run
WORK = os.path.join(cfg["paths"]["work_root"], "smoke" if SMOKE_TEST else "", "work05")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
from ultralytics import YOLO
import ultralytics, cv2
print("ultralytics", ultralytics.__version__, "| opencv", cv2.__version__)
_m = YOLO(cfg["model"]["weights"])
WEIGHTS = os.path.abspath(cfg["model"]["weights"])
NAMES = dict(_m.names)
del _m
run = V2Run(cfg, SMOKE_TEST, REPO)
print("prereg addendum sha256:", run.addendum_sha)

def zip_now(tag=""):
    zpath = f"/kaggle/working/results_05_controller_v2{'_smoke' if SMOKE_TEST else ''}{tag}.zip"
    base = os.path.dirname(RESULTS)
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for top in (RESULTS, LOGS):
            for root, _, files in os.walk(top):
                for fn in files:
                    full = os.path.join(root, fn)
                    z.write(full, os.path.relpath(full, base))
    print("wrote", zpath)

def stage(fn, *a):
    t = time.time()
    try:
        out = fn(*a)
    except Exception as e:
        print(f"\n*** {fn.__name__} FAILED: {e!r}\n*** writing partial results zip, then stopping ***")
        zip_now("_PARTIAL")
        raise
    print(f"{fn.__name__}: {(time.time()-t)/60:.1f} min | gates so far:",
          {k: ("PASS" if v["pass"] else "FAIL") for k, v in run.stages.items()})
    return out


In [ ]:
# ===================== CELL 4: STAGE 1 - ENGINES + CALIB CACHE =====================
S1 = stage(run.stage1_engines, WEIGHTS, NAMES)
print("microbench median engine ms:", {k: round(v["infer_ms"]["median"], 3) for k, v in S1["microbench"]["per_backend"].items()})
print("model-size rule:", S1["microbench"].get("size_rule"), "| switch penalty ms:", round(S1["switch_penalty"], 3))
for c in run.stages["1_engines_calib_cache"]["checks"]:
    print(("PASS " if c["pass"] else "FAIL ") + c["check"], c.get("detail", ""))


In [ ]:
# ===================== CELL 5: STAGE 2 - CALIB CONTROLLER DEV =====================
S2 = stage(run.stage2_calib_dev)
R = run.st.results
for n in ("fixed_fp16", "fixed_int8", f"adaptive_K{cfg['controller']['K']}", "random_matched", "oracle_cheap"):
    a = R[n]["all"]
    print(f"{n:18s} mAP50-95 {100*a['map50_95']:6.2f}  INT8 frac {a['frac_level'].get('int8', 0):.2f}")
print("feature timing:", run.st.feature_timing["featurize_ms"])


In [ ]:
# ===================== CELL 6: STAGE 3 - FIT + LOCK v2 (calib only) =====================
S3 = stage(run.stage3_v2)
print("model params:", S3["params"], "| held-out Spearman (calib CV):", round(S3["spearman"], 4))
for rho, v in S3["per_rho"].items():
    b = v["v2"]
    print(f"rho={rho}: K={b['K']} tau={b['tau']:.4f} calib INT8 {b['calib_frac_int8']:.3f} calib mAP50-95 {100*b['calib_map50_95']:.2f}"
          f" | v1 calib INT8 {v['v1_calib_frac']:.3f}")


In [ ]:
# ===================== CELL 7: STAGE 4 - TEST CACHE (only after a verified lock) =====================
S4 = stage(run.stage4_test_cache)
print(S4)


In [ ]:
# ===================== CELL 8: STAGE 5 - ENERGY UNDER MIXED SWITCHING =====================
S5 = stage(run.stage5_v2_energy)
import pandas as pd
rows = []
for k, v in S5["summary"].items():
    rows.append({"config": k, "lat med ms": v["latency_total_ms_median"]["median"], "fps": v["throughput_fps"]["median"] if v["throughput_fps"] else None,
                 "E_total mJ/frame": v["E_total_mJ_per_frame"]["median"] if v["E_total_mJ_per_frame"] else None,
                 "E_dyn mJ/frame": v["E_dyn_mJ_per_frame"]["median"] if v["E_dyn_mJ_per_frame"] else None,
                 "INT8 frac": v.get("frac_int8"), "switch/100": v.get("switch_rate_per_100")})
pd.set_option("display.width", 250)
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("live vs replay:", S5["live"])


In [ ]:
# ===================== CELL 9: STAGE 6 - TEST EVALUATION (once) =====================
S6 = stage(run.stage6_v2_eval)
for rho, h in S6["headroom"].items():
    print(f"rho={rho}: v2 {100*h['v2']:.2f} | random {100*h['random']:.2f} | v1 {100*h['v1']:.2f} | lagged {100*h['lagged_oracle']:.2f}"
          f" | oracle {100*h['oracle']:.2f} | captured {h['captured']}")
print(json.dumps({k: v for k, v in S6["hypotheses"].items() if k != "P4_headroom"}, indent=1, default=str)[:3000])


In [ ]:
# ===================== FINAL CELL: ZIP RESULTS =====================
import zipfile
suffix = "_smoke" if SMOKE_TEST else ""
zpath = f"/kaggle/working/results_05_controller_v2{suffix}.zip"
base = os.path.dirname(RESULTS)  # contains results/ and logs/
n = 0
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for top in (RESULTS, LOGS):
        for root, _, files in os.walk(top):
            for fn in files:
                full = os.path.join(root, fn)
                z.write(full, os.path.relpath(full, base))
                n += 1
print(f"wrote {zpath} ({n} files, {os.path.getsize(zpath)/1e6:.2f} MB)")
print(f"total notebook time: {(time.time()-T0)/60:.1f} min")
